# Can teacher training decompose a fine-tune with no trigger faithfully?

*E37, `experiments/teacher_spread`, 2026-10-26 and 2026-10-27. Record: `reports/analyses/2026-10-26_analysis_teacher-spread.md`,
with Amendment 1 as its last section. This notebook gives Amendment 1 first, then the original run in full. Follows
E34, `reports/analyses/2026-10-23_analysis_teacher-training.md`, and uses E36's lowercase fine-tune. The original run
was independently checked on 2026-10-26, `reports/reviews/2026-10-26_review_teacher-spread.md`; Amendment 1 on 2026-10-27,
`reports/reviews/2026-10-27_review_teacher-spread-amend1-analysis.md`. Corrections applied to both.*

**Words.** **The decomposition after teacher training**, or "the trained decomposition", is the released
decomposition with its alive subcomponents trained to imitate the fine-tuned model, Γ and Δ frozen. **The fine-tuned
model** is E36's lowercase fine-tune, $W_0 + \delta W$. The original run's cells below call the trained decomposition
"the copy".

## Amendment 1 (2026-10-27): train longer, start from a passing decomposition, or add VPD's masked losses

### The answer

Training longer does not bring the trained decomposition under the pass line, and neither does starting it from the
adjoint split. It reads 2.2 times the original decomposition's adversarial damage at step 1,000 and again at step
4,000, while its gap to the fine-tuned model keeps shrinking. Started from the adjoint split, which passes, training
crosses the fail line within a thousand steps and is level with the longer run by step 2,000. The masked-loss fix was
not run. Its control, the same recipe with
nothing to copy, raised the adversarial reading more than copying the fine-tune did.

### What we were trying to test

The original run, below, trained the subcomponents to imitate a fine-tune that lowercases Pile text, a change at every
position. The trained decomposition kept all of the fine-tune's gain on held-out lowercase text. But under the paper's
adversarial check it read 2.20 times the original decomposition, above the fail line of 1.61, and its loss was still
falling. The user chose to train longer and to try other fixes in the same job.

Every arm is **teacher training**: the alive $U$ and $V$ are trained to minimise $\mathrm{KL}(\text{fine-tuned}
\,\|\, \text{student})$ at every position, with Γ and Δ frozen. Each step uses eight of the original run's 2,400
lowercase rows, with Adam at rate 1e-5.

- **The longer run**, arm 1: the original recipe continued to 4,000 steps. It asks whether the original run was simply
  under-trained. Its first read, at step 1,000, must reproduce the original run.
- **The adjoint-start run**, arm 3: the same training, started from the scaled adjoint split, which keeps 0.80 of the
  gain and passes the check. It asks whether the failure belongs to where training starts or to where it has to end.
- **Masked teacher training**, arm 2: the same loss plus VPD's own two masked reconstruction losses. One draws random
  masks from Γ's importances, the other is a persistent mask adversary. These losses penalise exactly what the check
  probes.
- **The masked recipe's control**, run before arm 2: the masked recipe for 1,000 steps with the fine-tuned model
  replaced by the unchanged model, so there is nothing to copy. It must hold still, or a fall in arm 2 could be the
  losses lowering the check on any decomposition. One retry was allowed at a tenth of the masked losses' weights.

**What is read.** The **check ratio** is the adversary's KL on the state over its KL on the original decomposition, on
256 Pile rows: 1 is no more fragile than the original. It passes at or under 1.40 and fails at or over 1.61. **Behaviour
kept** is the share of the fine-tuned model's log-probability gain on held-out lowercase text, from 0 to 1. The **gap**
is $\mathrm{KL}(\text{fine-tuned} \,\|\, \text{state})$ on that text: 0.272 nats per token for the unchanged model.
**Damage elsewhere** is $\mathrm{KL}(W_0 \,\|\, \text{state})$ on plain Pile text: 0.235 for the fine-tuned model.

### What we expected, and why

- **The longer run still fails at step 4,000.** A ratio of 2.20 seemed too far above the line for more steps to close.
  The still-falling loss made this a medium-low confidence call.
- **The adjoint-start run ends where the longer run does**, with its check within a quarter of the longer run's. If
  the failure goes with the fine-tuned model's change on ordinary text, the start should not matter.
- **Masked teacher training passes with the behaviour kept**, at low-medium confidence, because its losses train on
  what the check reads.
- **The masked control holds still**, with a check between 0.7 and 1.4.

### Does training longer, or starting from the adjoint split, bring the check under the pass line?

![check and gap over training](../experiments/teacher_spread/figures/amend1_steps.png)

**Reading it.** No. In the top panel the longer run, blue, reads 2.20, 1.89 and 2.23 at its three reads, every one
above the red fail line. The adjoint-start run, purple, begins at 1.05, inside the unchanged model's band. By step
1,000 it has climbed to 1.76, a fifth below the longer run; by step 2,000 it is within 3% of it. The hollow circle is
the original run's second batch order, 1.88: the longer run moves between reads by about the batch-order spread. The
middle panel shows the gap to the fine-tuned model still shrinking on both runs, by a fifth to a quarter per doubling
of steps. The bottom panel shows the training KL still falling too. So the trained decomposition keeps getting closer
to the fine-tuned model without getting less fragile.

Along the way the behaviour stays kept, between 0.98 and 1.02 of the gain at every trained read. The damage on plain
Pile text sits at the fine-tuned model's level. The adjoint start passed at a fifth of that damage and failed by its
next read, once its damage matched. Damage is not the whole story, though: the masked control's retry, next section,
fails the check at 3.14 with a thirtieth of that damage.

**This outcome falls between the filed branches.** The tree sends the run on to 16,000 steps if both the training
loss and the check keep falling. It goes to its last branch if the loss is flat, falling under 10% per doubling. The loss fell
44% over the last doubling while the check did not fall. The run's code labels that "flat" by default. Taking the
last branch is our judgement, and a weak one: three reads with a 14% batch-order spread cannot tell a flat check from
a slow fall.

| state | steps | behaviour kept | damage | check |
|---|---|---|---|---|
| unchanged model | 0 | 0 | 0 | 1 |
| longer run | 1,000 / 2,000 / 4,000 | 0.998 / 0.982 / 1.000 | 0.234 / 0.242 / 0.243 | 2.20 / 1.89 / 2.23 |
| adjoint-start run | 0 / 1,000 / 2,000 | 0.804 / 1.020 / 0.990 | 0.050 / 0.237 / 0.244 | 1.05 / 1.76 / 1.84 |
| fine-tuned model left in Δ, original run | — | 1 | 0.235 | 4.87 |

### With nothing to copy, does the masked recipe leave the decomposition where it was?

![masked control](../experiments/teacher_spread/figures/amend1_m0.png)

**Reading it.** No, by a wide margin. The figure plots the check against damage on plain Pile text. The unchanged
model sits at the corner of the green box, the region the masked control had to stay in. After 1,000 steps with
nothing to copy, the masked recipe read 3.84, with damage of 0.021. The retry at a tenth of the weights cut the damage
to 0.007 but still read 3.14. Both sit above every decomposition trained on the fine-tuned model, which did ten times
the damage or more. So the masked recipe moves the decomposition even with nothing to copy, and is not usable as
specified. By the filed rule arm 2 was skipped, and its prediction is not scored.

The recipe's own losses rose rather than fell over those steps. The persistent adversary's loss went from 0.44 to
0.83, and the teacher KL from near 0 to 0.019. The same thing happened in E15, where Γ was retrained at the
unchanged model on a loss stack that includes these two losses, set up by the same code. Its check also left the
floor, and its adversary's loss also rose. Whether VPD's losses, our set-up of them, or a fault in the new trainer
moves the check is untested.

### Did the controls behave?

Yes. The unchanged model read its known noise floor, 0.0748, and the same values as in the original run. The
regenerated fine-tuned model reproduced E36's gain, 0.1300 nats per token against 0.1301. The longer run's first read
reproduced the original run, and the adjoint-start run's first read reproduced the original adjoint split. The masked
trainer's first loss matched the plain trainer's, so its control's rise did not come from a mismatch in the first
step's loss. Its forward pass still differs from the plain one by 0.0007 nats at the unchanged model. Under Adam, a
small steady gradient from that difference could drive drift; a control with the masked weights at zero would test it.

### What we learned

On this broad fine-tune, teacher training lands on a decomposition about twice as fragile as the original under the
paper's check. Training four times longer does not change that, and neither does starting from a decomposition that
passes. Training brings the trained decomposition closer to the fine-tuned model, and that does not bring the check
down with it. The one fix aimed at fragility itself could not be tested, because its recipe raised the check at the
unchanged model, as E15's recipe had.

### How compelling is the evidence

- **Blind spots.** The check is one adversary from one random start, a lower bound, as the user fixed it on
  2026-10-27. Behaviour kept and the gap are means over every position; damage is read on plain Pile text only. The
  masked control's verdict does not say which part of the recipe moved the check.
- **Noise.** The unchanged model's seed spread is 0.075. The original run's two batch orders differed by 14% in the
  check, and the longer run's reads span 18%. Every trained read is at least 9% above the fail line.
- **Scope.** One fine-tune, one batch order, one adversary seed, one synthetic style, one adjoint start.

### Surprises

- **The masked recipe's control moved far off the unchanged model.** We expected a check between 0.7 and 1.4 and read
  3.84, and 3.14 at a tenth of the weights, with the recipe's own losses rising. We have no tested cause.
- **The longer run's outcome is not covered by the filed tree.** Its training loss fell 44% over the last doubling;
  the plan calls a run flat below 10%. Yet the check did not fall. The run's code reported "flat" by default.

### What next

No arm passed, and the longer run falls between the filed branches. We read it, as a judgement, as the tree's last
branch: a card to the user with option (d) and the anchor question, with extending added as an option.

1. **Extend the longer run to 16,000 steps**, about $1.5. A check falling under the fail line would put
   under-training back as the explanation; a check still near 2 would close it.
2. **Option (d): does the frozen Γ limit the trained decomposition, or the subcomponents' span?** The Γ test is E15's
   refit, about $9 per state. Training against our set-up of VPD's losses at the unchanged model has now failed three
   times. So a control with the masked losses' weights at zero comes first, about $1. It would say whether the masked
   terms or the rest of the loop move the subcomponents off the floor.
3. **The anchor question for broad updates.** Every trained read here is under half of the fine-tuned model left in
   Δ. Judged against that state, all pass; by the fixed line, all fail. A cheap control would inform the choice: move
   the subcomponents at random by the retry's size and read the check. The choice sets what the larger model is scored
   against.
4. **The larger model with teacher training as it is**, its check reported both ways. Another reading of about twice
   would make this the norm for broad updates rather than a quirk of a toy fine-tune.

### Scorecard, Amendment 1

Filed 2026-10-27 11:12:18 in `experiments/teacher_spread/results/amend1_predictions.json`, unedited; scored by the
run's own `amend1.score`.

| id | claim | read | verdict |
|---|---|---|---|
| P1 | arm 1 at step 4,000 still fails, check ≥ 1.61 | 2.227 | **holds** |
| P2 | arm 2 at step 2,000 passes with s ≥ 0.9 | not run: M0 and M0b failed | **not scored** |
| P3 | arm 3 ends within 25% of arm 1 at step 2,000, s ≥ 0.9 | 1.836 against 1.894; s 0.990 | **holds** |
| M0 | masked recipe at W₀: damage ≤ 0.02, check ≤ 1.40 | 0.021, 3.84; retry 0.007, 3.14 | **fail** |
| R1 | arm 1 at step 1,000 reproduces E37 | s 0.998, check 1.0005 × E37 | reproduced |

C1, C2, C3 and G2 passed; arm 3's start reproduced E37's adjoint split.

### Not run, Amendment 1

Arm 2, masked teacher training on the fine-tuned model. The masked control with the masked weights at zero. A random
move of the subcomponents by the retry's size, read under the check. A sweep of
the masked weights beyond one retry; Γ trained with the subcomponents. The longer run past 4,000 steps or the
adjoint-start run past 2,000; either at a second batch order. A second adversary seed or start on any trained state. A
second broad fine-tune. The CPU cosine between the longer run's step-1,000 offsets and the original run's. The larger
model.

### Provenance, Amendment 1

RunPod H100 80GB HBM3; job 12:18 to 12:48 after a 3-minute load, exit 0, $2.27. Results
`experiments/teacher_spread/results/amend1.json`, sha256 `6d4c1b5c…`; logs `results/large/amend1_run.log` and
`amend1_stdout.log`. Figures from `experiments/teacher_spread/plot_amend1.py`. Code `amend1.py`,
`src/commute/masked_teacher.py`.

## The original run (2026-10-26)

Everything below is E37 as first run and checked: teacher training read at step 1,000 only.


> **Something unexpected happened that I cannot explain.** We predicted the lowercase copy would pass the adversarial
> check, as E34's copies of trigger fine-tunes had. It read 2.20 times the unchanged model, above the fail line of
> 1.61. A second filed prediction, that the adjoint split would lose most of the behaviour, also failed. That one
> drew on a different measure in E34, so it is a smaller surprise than it looks.

### The answer

Not as filed. Teacher training copied the lowercase fine-tune almost exactly: the copy keeps all of its held-out gain
and does the same damage elsewhere. But the standard mask adversary hurts it twice as much as the unchanged model,
past the fail line, at both batch orders and from ordinary text. The copy was still learning when read. The adjoint
split kept most of the gain and passed, but it barely changes predictions on ordinary text, which may be why it passes.

### What we were trying to test

**The model.** The released 67M Pile model with its VPD decomposition. Each matrix is $W = UV^\top + \Delta$; the
columns of $U$ and $V$ are the alive subcomponents' write and read vectors. Γ and Δ are frozen throughout.

**The teacher.** E36's lowercase fine-tune. All 24 decomposed matrices were trained on the next-token loss at every
position of lowercased Pile text, where capitals never come. Adam at rate 3e-5, 300 steps. Its update is
$\delta W$. Every fine-tune that teacher training copied before acted at one trigger token; this one changes
predictions everywhere, as post-training of a larger model would.

**Teacher training** trains the alive $U$ and $V$ to minimise $\mathrm{KL}(\text{teacher} \,\|\, \text{student})$ on
unlabelled text, Γ and Δ frozen, for 1,000 steps with E34's recipe.

**Six states**, each with the alternative it rules out:

- the **unchanged model**, the floor every ratio is taken against;
- **do-nothing**: $\delta W$ goes into Δ, the subcomponents stay as released. The reference: any map should do at
  least this well;
- the **adjoint split** of $\delta W$, the cheap closed-form map, which lost the trigger behaviours in E34;
- the **lowercase copy**, trained on the lowercase rows the teacher trained on;
- the **plain-text copy**, the same documents with capitals kept: is ordinary text enough to copy a style?
- the **lowercase copy at a second batch order**: the copy's noise, and whether two copies land on the same weights.

**What is read**, one symbol per line:

- **behaviour kept** $s$: the state's gain in mean log-probability on held-out lowercase text, over the teacher's.
  0 is the unchanged model, 1 the teacher;
- **teacher gap share**: $\mathrm{KL}(\text{teacher} \,\|\, \text{state})$ over the unchanged model's. 0 is the teacher;
- **damage**: $\mathrm{KL}(W_0 \,\|\, \text{state})$ on plain Pile text, how much the state changes ordinary text;
- **adversarial ratio**: an adversary searches 20 steps for the masks that pull the masked model furthest from the
  target. The ratio is its KL over the unchanged model's; 1 is no more fragile;
- **cosine** between two weight changes, over all 24 matrices at once.

The adversarial check passes at or under 1.40 and fails at or over 1.61 on this card; between is inconclusive. The
adversary may also switch off Δ. In do-nothing Δ holds the whole update, so do-nothing's ratio is high by construction.

### What we expected, and why

- **The lowercase copy keeps at least 0.9 of the gain** and leaves at most a tenth of the gap. Every E34 copy
  reached its teacher.
- **It passes the adversarial check.** All of E34's Adam copies did, even where their teachers did not.
- **Its damage on plain text is half to twice the teacher's**: a faithful copy shares the side effects.
- **The adjoint split keeps under half the gain.** On five of E34's six trigger teachers it kept a quarter or less of
  the target probability, a probability rather than a share of log-probability gain.
- **The plain-text copy also keeps the behaviour**, at low confidence.
- **Two batch orders land on the same weight change**, a cosine of at least 0.8; under 0.5 would mean one of many.

### What we observed

#### Does the copy keep the behaviour, and is it faithful?

![behaviour and faithfulness](../experiments/teacher_spread/figures/behaviour_faithful.png)

**Reading it.** It keeps the behaviour and fails the adversarial check. Both lowercase copies, in blue, sit at the
right edge with the teacher, keeping 0.998 and 0.994 of the gain. Both sit above the red fail line, at 2.20 and 1.88.
The plain-text copy, orange, keeps 0.92 and also fails, at 1.74. The adjoint split, grey, keeps 0.80 of the gain and
sits at 1.05, inside the unchanged model's own spread. Do-nothing, black, reads 4.87, high by construction.

The copy is close to the teacher in every other way. It leaves 0.04 of the gap and does the teacher's damage on plain
Pile text. The adjoint split leaves 0.41 of the gap and a fifth of the damage. So the adjoint split raises the
log-probability of lowercase text without matching the teacher's predictions, while the copy matches them.

**An alternative reading, untested.** The ratio rises with damage on ordinary text. The adjoint split changes Pile
predictions by 0.05 nats and passes; the copies change them by 0.23 nats, as the teacher does, and fail. This
teacher changes ordinary text several times more than E34's did. Relative to its own teacher the copy reads 0.39 to
0.45 here, against about 0.6 to 1.2 for E34's copies. By that measure this copy is no worse than E34's.

Judged against do-nothing, as the anchor card's option b would, every copy passes, but that flatters every map.

The copies were still learning: their training KL fell by a third between steps 500 and 1,000. The filed decision tree,
read literally, sends this outcome to "under-trained, rerun longer". The 2.20 is a reading of an unfinished copy at
one step.

| state | behaviour kept | gap share | damage, nats | adversarial |
|---|---|---|---|---|
| do-nothing | 1 | 0 | 0.235 | 4.87 |
| adjoint split | 0.80 | 0.41 | 0.050 | 1.05 |
| lowercase copy | 0.998 | 0.04 | 0.234 | 2.20 |
| lowercase copy, 2nd order | 0.994 | 0.04 | 0.234 | 1.88 |
| plain-text copy | 0.92 | 0.08 | 0.224 | 1.74 |

#### Does the failure sit in one half of the rows?

![blocks](../experiments/teacher_spread/figures/blocks.png)

**Reading it.** No: both halves put the lowercase copies over the fail line. The harness averages its adversary over
two blocks of 128 rows. The lowercase copy reads 1.69 on the first block and 2.68 on the second. The plain-text copy
reads 1.60 on the first, just under the fail line, and 1.86 on the second. The hollow diamonds are the budget reading,
the same search on the first block's rows, read in full precision. It agrees with the first block within about a
tenth for every state. So it is not a second adversary: this run has one adversary at one seed per state.

#### Do two copies land on the same weight change?

![placement](../experiments/teacher_spread/figures/placement.png)

**Reading it.** Yes. The two batch orders' weight changes have a cosine of 0.93, above the 0.8 line, and the lowest
matrix reads 0.91. The lowercase and plain-text copies read 0.81 against each other. Each copy points along the
fine-tune at 0.44, level with the adjoint split. E34's copies read 0.08 to 0.29 against their fine-tunes. A random
direction reads zero. Between batch orders the behaviour kept moved by 0.004. The adversarial ratio moved by 14%,
inside the spread E34 saw.

### What we learned

Teacher training can copy a broad fine-tune: the copy matches the teacher's predictions to a twentieth of the
original gap and shares its side effects. But at step 1,000 the copy is not faithful by the fixed line, at two batch
orders and from two kinds of text, under one adversary. Relative to its teacher it is no worse than E34's copies were;
this teacher simply changes ordinary text far more, and the ratio rises with that change. Whether that explains the
failure is untested. The adjoint split keeps most of a broad gain and passes, while barely changing ordinary text.

### How compelling is the evidence

- **One teacher, one fine-tune seed, one synthetic style**, read at one step on copies still learning.
- **One adversary, one seed per state.** The budget reading repeats the first block. What was varied is batch order
  and kind of text, and all three copies fail.
- **Seed noise.** A copy's batch-order spread in the ratio is 14% here and up to 25% in E34. The lowercase copy sits
  37% above the fail line; the plain-text copy only 8% above it.
- **Do-nothing's reading is high by construction**, so every comparison with it flatters the map.
- **Behaviour kept is a mean over positions.** The adjoint split shows the mean can be kept with different
  predictions.
- **The damage reading rests on three damage levels**; it is a reading, not a test.

### Surprises

- **The copy fails the adversarial check, against a filed prediction that it would pass.** All of E34's Adam copies
  had read under 1.37. We have no tested cause. The copy was still learning, the update is broad, and the teacher
  changes ordinary text several times more than E34's did.
- **The adjoint split keeps 0.80 of the gain, against a filed prediction of under half.** The verdict stands by the
  filed rule. But the expectation came from E34's target probabilities, another measure. Converted to shares of
  log-probability gain, the independent check estimates E34's adjoint split kept about 0.5 to 0.6.
- **The copies point along the fine-tune more than E34's did**, 0.44 against an expected 0.08 to 0.3.
- **The copies are three quarters of the teacher's size**, not the expected half.

### What next

The options on card `decision-broad-copy`, in its order:

1. **Train the copy three times longer: the filed tree's own branch, and the cheapest step.** About $2, reading the
   adversarial check at several steps and two adversary seeds. A ratio falling under the fail line as the loss
   flattens means under-training; one that stays means the failure is the copy's.
2. **Judge the copy against do-nothing**, option b of the anchor card. Every copy here then passes, but do-nothing's
   reading is high by construction.
3. **Take the adjoint split as the map for broad updates, or start the copy from it.** A copy started there and read
   at several steps would show whether matching the teacher's changes on ordinary text brings the fragility.
4. **If the first step leaves the failure standing, test whether the frozen Γ or the subcomponents' span limits the
   copy**, E15's refit at about $9.

### Scorecard

| id | claim | read | verdict |
|---|---|---|---|
| P1 | lowercase copy keeps the behaviour, s ≥ 0.9 | 0.998 | holds |
| P2 | lowercase copy close to the teacher, gap share ≤ 0.10 | 0.044 | holds |
| P3 | lowercase copy passes the harness, ≤ 1.40 | 2.20, fail line 1.61 | **fails, reversal** |
| P4 | damage 0.5 to 2 times the teacher's | 0.996 | holds |
| P5 | adjoint split loses the behaviour, s < 0.5 | 0.804; expectation from another measure | **fails** |
| P6 | plain-text copy keeps the behaviour, s ≥ 0.9 | 0.919 | holds |
| P7 | one placement, cosine ≥ 0.8 | 0.929 | holds |

All three controls passed. The unchanged model read its known floor, 0.0748. The teacher reproduced E36's gain within
0.00015, and do-nothing read exactly as the teacher.

### Not run

- A second fine-tune seed or a second broad teacher.
- Copies read at other steps, or trained longer.
- The plain-text copy at a second batch order; the second batch order's budget reading.
- A second adversary seed on any updated state; an independent second adversary; adversaries past 20 steps.
- A test of the damage reading: maps with matched damage on ordinary text.
- A copy started from the adjoint split; E15's Γ refit.
- LoRA teachers on this task; the larger model.

**Provenance.** RunPod H100 80GB HBM3, about 21 min and $1.40, job 20:22 to 20:36, exit 0. Results
`experiments/teacher_spread/results/teacher_spread.json`; predictions `results/predictions.json`, filed 19:53:47,
unedited; figures from `experiments/teacher_spread/plot.py`. Full tables, block readings and hashes are in the report.